#**Projet** — Résumé automatique extractif avec attention

In [1]:
!pip install --quiet datasets transformers nltk scikit-learn rouge-score torch tqdm sentencepiece

  Preparing metadata (setup.py) ... done


## Chargement d'un sous-ensemble du dataset CNN/DailyMail

Dans cette étape, nous chargeons un petit sous-ensemble du dataset CNN/DailyMail afin de pouvoir développer et tester notre pipeline rapidement.  

L'objectif est de se familiariser avec la structure des données et de vérifier leur qualité avant tout prétraitement.  

Chaque élément du dataset contient :  
- **Article** : le texte complet de l'article de presse.  
- **Highlights** : le résumé ou points clés de l'article, fournis par les auteurs.  

Nous examinons ensuite un exemple pour voir :  
- la longueur de l'article,  
- et un extrait du résumé (highlights).  

Cette étape permet de mieux comprendre la nature des données avant de segmenter les articles en phrases et de préparer les labels pour le résumé extractif.


In [2]:
import os
import random
import math
from datasets import load_dataset
import nltk
nltk.download('punkt')


import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from rouge_score import rouge_scorer


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


# Fix seeds
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Device: cpu


In [3]:
# Charge un petit sous-ensemble pour développement
ds_name = 'cnn_dailymail'
version = '3.0.0'
raw = load_dataset(ds_name, version)
print(raw)


# Exemple d'accès
example = raw['train'][0]
print('Article length chars:', len(example['article']))
print('Highlights:', example['highlights'][:300])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})
Article length chars: 2527
Highlights: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


## Résultats du chargement du dataset CNN/DailyMail

Après le chargement, nous obtenons les informations suivantes :

- **Taille des splits :**
  - Train : 287 113 articles
  - Validation : 13 368 articles
  - Test : 11 490 articles

- **Structure des données :**
  - Chaque entrée contient trois champs : `article`, `highlights` (résumé), et `id`.

- **Exemple d'article :**
  - Longueur de l'article : 2 527 caractères
  - Extrait du résumé (highlights) :
    - « Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday. Young actor says he has no plans to fritter his cash away. Radcliffe's earnings from first five Potter films have been held in trust fund. »

> Ces résultats confirment que le dataset est correctement chargé et prêt pour le prétraitement et la segmentation en phrases.


# Prétraitement de l'article

Cette cellule prépare le texte pour l'analyse NLP en suivant ces étapes :

1. **Tokenisation en phrases**  
   - Découpe le texte en phrases.  
   - Supprime les phrases vides ou composées uniquement d'espaces.

2. **Limitation du nombre de phrases**  
   - Conserve seulement les `max_sentences` premières phrases pour éviter les textes trop longs.

3. **Nettoyage des phrases**  
   - Chaque phrase est découpée en mots.  
   - Suppression des **stopwords** (mots très fréquents et peu informatifs).  
   - Suppression de la ponctuation.  
   - Reconstruction de la phrase nettoyée.

4. **Résultat**  
   - Retourne une liste de phrases prétraitées, prêtes pour des analyses NLP (extraction de mots-clés, vectorisation, etc.).

**Exemple d'utilisation :**  
- Nettoyer un article pour obtenir des phrases épurées et limitées, facilitant le traitement automatique.


In [4]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
import string

# Télécharger les ressources nécessaires
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # add this line to download the missing resource

stop_words = set(stopwords.words('english'))  # ou 'french' selon ton texte

def preprocess_article(article, max_sentences=50):
    # Tokenize en phrases
    sents = sent_tokenize(article)
    sents = [s.strip() for s in sents if len(s.strip()) > 0]

    # Limiter le nombre de phrases
    if len(sents) > max_sentences:
        sents = sents[:max_sentences]

    # Nettoyage des stopwords et ponctuations pour chaque phrase
    cleaned_sents = []
    for sent in sents:
        words = word_tokenize(sent)
        words = [w for w in words if w.lower() not in stop_words and w not in string.punctuation]
        cleaned_sents.append(' '.join(words))

    return cleaned_sents

# Test
sents = preprocess_article(example['article'], max_sentences=50)
print('Nombre de phrases:', len(sents))
print(sents[:5])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Nombre de phrases: 24
["LONDON England Reuters -- Harry Potter star Daniel Radcliffe gains access reported £20 million 41.1 million fortune turns 18 Monday insists money wo n't cast spell", "Daniel Radcliffe Harry Potter `` Harry Potter Order Phoenix '' disappointment gossip columnists around world young actor says plans fritter cash away fast cars drink celebrity parties", "`` n't plan one people soon turn 18 suddenly buy massive sports car collection something similar '' told Australian interviewer earlier month", "`` n't think 'll particularly extravagant", "`` things like buying things cost 10 pounds -- books CDs DVDs ''"]


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Sélection des phrases importantes avec TF-IDF

Cette cellule permet de **sélectionner les phrases les plus importantes** d'un texte en utilisant la pondération TF-IDF.

## Étapes principales :

1. **Vectorisation TF-IDF**  
   - Chaque phrase est convertie en vecteur TF-IDF.  
   - Les **stopwords** en anglais sont ignorés pour ne garder que les mots informatifs.

2. **Calcul du score des phrases**  
   - Le score d'une phrase est la **somme des poids TF-IDF** de ses mots.  
   - Plus le score est élevé, plus la phrase est considérée comme importante.

3. **Sélection des phrases les plus importantes**  
   - On sélectionne les `k` phrases ayant les scores les plus élevés.  
   - Les phrases sélectionnées sont ensuite **réordonnées selon leur position originale** dans le texte pour garder le sens.

## Résultat :

- `selected` : liste des phrases sélectionnées.  
- `idx_sorted` : indices originaux de ces phrases dans le texte.

**Exemple d'utilisation :**  
- Identifier les 3 phrases les plus représentatives d'un article pour résumer ou extraire l'information clé.


In [5]:
vectorizer = TfidfVectorizer(stop_words='english')


def tfidf_select(sents, k=3):
  if len(sents) == 0:
    return []
  X = vectorizer.fit_transform(sents)
  scores = X.sum(axis=1).A1
  idx = np.argsort(scores)[-k:][::-1]
  # Conserver dans l'ordre original
  idx_sorted = sorted(idx)
  selected = [sents[i] for i in idx_sorted]
  return selected, idx_sorted


# Test
selected, idxs = tfidf_select(sents, k=3)
print('Indices sélectionnés:', idxs)
print(''.join(selected))

Indices sélectionnés: [np.int64(0), np.int64(1), np.int64(5)]
LONDON England Reuters -- Harry Potter star Daniel Radcliffe gains access reported £20 million 41.1 million fortune turns 18 Monday insists money wo n't cast spellDaniel Radcliffe Harry Potter `` Harry Potter Order Phoenix '' disappointment gossip columnists around world young actor says plans fritter cash away fast cars drink celebrity parties18 Radcliffe able gamble casino buy drink pub see horror film `` Hostel Part II '' currently six places number one movie UK box office chart


#Construction du vocabulaire à partir de plusieurs articles

Pour entraîner notre modèle RNN, il est important de construire un vocabulaire représentatif d'un ensemble d'articles, et pas seulement d'un seul article.  

- Chaque article est découpé en phrases (`sent_tokenize`) et chaque phrase est tokenisée en mots (`word_tokenize`).  
- Les mots rares (fréquence < `min_freq`) sont ignorés pour limiter la taille du vocabulaire.  
- Deux tokens spéciaux sont ajoutés :  
  - `<PAD>` : utilisé pour le padding des séquences.  
  - `<UNK>` : pour les mots inconnus.  

Les dictionnaires générés sont :  
- `word2idx` : mot → index  
- `idx2word` : index → mot  

Ce vocabulaire sera utilisé pour **convertir les phrases en séquences d’indices**, prêtes à être utilisées par le modèle RNN.



In [6]:
from collections import Counter


def build_vocab_from_articles(articles, min_freq=2, max_sentences_per_article=50):
    """
    Construit le vocabulaire à partir d'une liste d'articles (texte brut).

    articles : list de textes
    min_freq : fréquence minimale pour inclure un mot
    max_sentences_per_article : limite du nombre de phrases par article pour dev
    """
    counter = Counter()
    for article in articles:
        # Tokenize en phrases
        sents = sent_tokenize(article)[:max_sentences_per_article]
        for sent in sents:
            tokens = word_tokenize(sent.lower())
            counter.update(tokens)

    # Filtrer les mots rares
    vocab = {word for word, freq in counter.items() if freq >= min_freq}

    # Créer dictionnaires
    word2idx = {w: i+2 for i, w in enumerate(sorted(vocab))}
    word2idx['<PAD>'] = 0
    word2idx['<UNK>'] = 1
    idx2word = {i: w for w, i in word2idx.items()}

    return word2idx, idx2word

# Exemple : utiliser les 100 premiers articles du train pour dev
train_articles = [raw['train'][i]['article'] for i in range(100)]
word2idx, idx2word = build_vocab_from_articles(train_articles, min_freq=5)
vocab_size = len(word2idx)
print('Taille vocabulaire réel:', vocab_size)


Taille vocabulaire réel: 1762


#Encodage des phrases et préparation du DataLoader

Pour entraîner le modèle RNN, chaque phrase des articles doit être convertie en **séquence d’indices** à partir du vocabulaire (`word2idx`):

- Les phrases plus courtes sont **complétées avec le token `<PAD>`** pour atteindre une longueur maximale `max_len`.
- Les mots inconnus sont remplacés par le token `<UNK>`.
- Chaque article est limité à `max_sentences` phrases.  
  - Si un article contient moins de phrases, on ajoute des phrases `<PAD>` pour atteindre exactement `max_sentences`.
  - Si un article contient plus de phrases, on ne conserve que les `max_sentences` premières.
  
Ensuite, nous créons un **Dataset PyTorch** qui contient toutes les phrases encodées et un **DataLoader** pour générer des batches de taille `batch_size`.  

- Chaque batch aura une forme `(batch_size, max_sentences, max_len)`.
- Cette structure uniforme permet d’entraîner facilement le modèle RNN avec attention.


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import sent_tokenize, word_tokenize

# Fonction pour encoder une phrase en indices
def encode_sentence(sentence, word2idx, max_len=20):
    tokens = word_tokenize(sentence.lower())
    seq = [word2idx.get(w, word2idx['<UNK>']) for w in tokens]
    if len(seq) < max_len:
        seq += [word2idx['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

# Dataset PyTorch avec padding des phrases
class SummaryDataset(Dataset):
    def __init__(self, articles, word2idx, max_len=20, max_sentences=50):

        self.data = []
        pad_sentence = [word2idx['<PAD>']] * max_len
        for article in articles:
            # Tokenize en phrases et limiter le nombre de phrases
            sents = sent_tokenize(article)[:max_sentences]
            # Encoder chaque phrase
            encoded_sents = [encode_sentence(s, word2idx, max_len) for s in sents]
            # Padding pour que chaque article ait exactement max_sentences phrases
            while len(encoded_sents) < max_sentences:
                encoded_sents.append(pad_sentence)
            self.data.append(torch.tensor(encoded_sents, dtype=torch.long))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Exemple : utiliser les 10 premiers articles pour le DataLoader
train_articles_subset = [raw['train'][i]['article'] for i in range(10)]
dataset = SummaryDataset(train_articles_subset, word2idx, max_len=20, max_sentences=50)

# Créer un DataLoader
loader = DataLoader(dataset, batch_size=2, shuffle=True, drop_last=True)

# Vérification
for batch in loader:
    print(batch.shape)  # (batch_size, max_sentences, max_len)
    break


torch.Size([2, 50, 20])


#Encodeur RNN avec attention Bahdanau

Pour améliorer la sélection des phrases clés dans le résumé, nous allons utiliser un **encodeur RNN** (GRU ou LSTM) combiné avec une **attention de Bahdanau** :

1. **Encodeur RNN**  
   - Chaque phrase (séquence de tokens) est passée dans un **embedding layer**.  
   - Les embeddings sont ensuite traités par un **GRU** pour produire une représentation cachée de chaque phrase.  
   - L’encodeur retourne **toutes les sorties cachées** (pour chaque token) et le **dernier état caché**.

2. **Attention Bahdanau**  
   - À chaque étape du décodeur, l’attention calcule un **score pour chaque sortie cachée de l’encodeur**.  
   - Ces scores sont transformés en **poids d’attention** via softmax.  
   - Le **context vector** est obtenu en faisant la somme pondérée des sorties cachées, selon les poids d’attention.  
   - Ce context vector permet au décodeur de se concentrer sur les parties importantes de l’article.

3. **Décodeur RNN (extractif)**  
   - Le décodeur prend le context vector et l’état caché pour **prédire quelles phrases doivent être incluses dans le résumé**.  
   - Les prédictions peuvent être comparées aux phrases sélectionnées par TF-IDF pour entraîner le modèle.

**Avantages :**  
- L’attention permet au modèle de se concentrer sur les phrases importantes.  
- L’encodeur RNN capture les dépendances entre les mots et phrases.  
- On peut comparer les résumés générés **avant et après attention** pour voir l’amélioration.




In [8]:

# --- Encodeur RNN ---
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True, bidirectional=True)
        self.hidden_size = hidden_size

    def forward(self, x):
        """
        x : (batch_size, max_sentences, max_len)
        """
        batch_size, max_sentences, max_len = x.size()
        # On combine les phrases pour traiter toutes les phrases en même temps
        x = x.view(batch_size * max_sentences, max_len)
        embedded = self.embedding(x)  # (batch*sentences, max_len, embed_size)
        outputs, hidden = self.gru(embedded)  # outputs : (batch*sentences, max_len, 2*hidden)
        # On récupère le dernier état de chaque phrase
        hidden = outputs[:, -1, :]  # (batch*sentences, 2*hidden)
        hidden = hidden.view(batch_size, max_sentences, -1)  # (batch, max_sentences, 2*hidden)
        return hidden  # Représentation de chaque phrase

# --- Attention Bahdanau ---
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W1 = nn.Linear(hidden_size*2, hidden_size)  # car bidirectionnel
        self.W2 = nn.Linear(hidden_size*2, hidden_size)
        self.V = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_outputs):
        """
        hidden : (batch, hidden*2) -> état du décodeur ou phrase précédente
        encoder_outputs : (batch, max_sentences, hidden*2)
        """
        hidden = hidden.unsqueeze(1)  # (batch, 1, hidden*2)
        score = self.V(torch.tanh(self.W1(encoder_outputs) + self.W2(hidden)))  # (batch, max_sentences, 1)
        attention_weights = torch.softmax(score, dim=1)  # (batch, max_sentences, 1)
        context_vector = torch.sum(attention_weights * encoder_outputs, dim=1)  # (batch, hidden*2)
        return context_vector, attention_weights

# --- Exemple d'utilisation ---
vocab_size = len(word2idx)
embed_size = 128
hidden_size = 256

encoder = EncoderRNN(vocab_size, embed_size, hidden_size).to(DEVICE)
attention = BahdanauAttention(hidden_size).to(DEVICE)

# Récupérer un batch
for batch in loader:
    batch = batch.to(DEVICE)  # (batch_size, max_sentences, max_len)
    encoder_outputs = encoder(batch)  # (batch, max_sentences, 2*hidden)
    # Exemple : prendre la 1ère phrase comme état caché du décodeur
    hidden = encoder_outputs[:, 0, :]  # (batch, 2*hidden)
    context, attn_weights = attention(hidden, encoder_outputs)
    print("Context vector shape:", context.shape)
    print("Attention weights shape:", attn_weights.shape)
    break


Context vector shape: torch.Size([2, 512])
Attention weights shape: torch.Size([2, 50, 1])


#Comparaison TF-IDF vs RNN+Attention avec ROUGE

Dans cette étape, nous générons des résumés extractifs pour un article et comparons les performances de deux méthodes :

1. **TF-IDF baseline**  
   - On sélectionne les `k` phrases les plus représentatives selon le score TF-IDF.

2. **RNN + Attention Bahdanau**  
   - L'encodeur RNN bidirectionnel transforme chaque phrase en un vecteur.  
   - L’attention de Bahdanau calcule un vecteur de contexte pour chaque phrase.  
   - Le décodeur extractif prédit un score par phrase, et les `k` phrases les mieux scorées sont sélectionnées.

3. **Prétraitement**  
   - Les phrases sont tokenisées et nettoyées (`preprocess_article`).  
   - Le nombre réel de phrases est utilisé pour **ignorer les `<PAD>`** lors de la sélection des phrases.

4. **Calcul ROUGE**  
   - On compare les résumés générés à la référence (`highlights`) avec les métriques ROUGE-1, ROUGE-2 et ROUGE-L.  

Ce processus permet d’évaluer qualitativement et quantitativement l’amélioration apportée par l’attention sur le résumé extractif.


### **La métrique ROUGE**

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) sert à évaluer automatiquement la qualité des résumés. Elle compare le texte généré à une ou plusieurs références humaines en mesurant le recouvrement des unités de texte (mots, n-grammes, séquences).

* ROUGE-1 : recouvrement des unigrammes (mots).

* ROUGE-2 : recouvrement des bigrammes (paires de mots).

* ROUGE-L : plus longue sous-séquence commune (évalue la qualité structurelle).

Chaque score est calculé à partir de précision, rappel et F-mesure.


-> Plus le score ROUGE est élevé, plus le résumé automatique est proche du résumé de référence.

In [9]:
# --- Décodeur Extractif ---
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = BahdanauAttention(hidden_size)
        self.out = nn.Linear(hidden_size*2, 1) # Pour prédire un score par phrase

    def forward(self, encoder_outputs):
        """
        encoder_outputs : (batch, max_sentences, hidden*2)
        """
        scores = self.out(encoder_outputs).squeeze(-1) # (batch, max_sentences)
        return scores

# --- Initialisation du décodeur ---
decoder = DecoderRNN(hidden_size).to(DEVICE)

In [10]:
from nltk.tokenize import sent_tokenize
import torch

# --- Paramètres ---
k = 3  # nombre de phrases à sélectionner
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# --- Fonction ROUGE ---
def compute_rouge(pred_sents, ref_sents):
    pred_text = " ".join(pred_sents)
    ref_text = " ".join(ref_sents)
    scores = scorer.score(ref_text, pred_text)
    return {k: v.fmeasure for k, v in scores.items()}

# --- Exemple sur le 1er article ---
example_article = train_articles_subset[0]

# Prétraitement
sents = preprocess_article(example_article, max_sentences=50)

# TF-IDF baseline
selected_tfidf, _ = tfidf_select(sents, k=k)

# RNN + Attention
# Récupérer batch pour 1 article
batch = dataset[0].unsqueeze(0).to(DEVICE)  # (1, max_sentences, max_len)
encoder_outputs = encoder(batch)
scores = decoder(encoder_outputs)  # (1, max_sentences)

# Nombre réel de phrases
num_real_sents = len(sents)

# Top-k indices valides
topk_indices = torch.topk(scores, k, dim=1).indices[0].cpu().numpy()
topk_indices_valid = [i for i in topk_indices if i < num_real_sents]

selected_rnn = [sents[i] for i in topk_indices_valid]

# Référence (highlights)
ref = raw['train'][0]['highlights'].split('. ')
ref = [r.strip() for r in ref if len(r.strip()) > 0]

# Calcul ROUGE
rouge_tfidf = compute_rouge(selected_tfidf, ref)
rouge_rnn = compute_rouge(selected_rnn, ref)

print("Résumé TF-IDF :", selected_tfidf)
print("\nRésumé RNN+Attention :", selected_rnn)
print("\nROUGE TF-IDF :", rouge_tfidf)
print("\nROUGE RNN+Attention :", rouge_rnn)


Résumé TF-IDF : ["LONDON England Reuters -- Harry Potter star Daniel Radcliffe gains access reported £20 million 41.1 million fortune turns 18 Monday insists money wo n't cast spell", "Daniel Radcliffe Harry Potter `` Harry Potter Order Phoenix '' disappointment gossip columnists around world young actor says plans fritter cash away fast cars drink celebrity parties", "18 Radcliffe able gamble casino buy drink pub see horror film `` Hostel Part II '' currently six places number one movie UK box office chart"]

Résumé RNN+Attention : ['Despite growing fame riches actor says keeping feet firmly ground', "Earlier year made stage debut playing tortured teenager Peter Shaffer 's `` Equus ''", "`` try hard go way would easy ''"]

ROUGE TF-IDF : {'rouge1': 0.33043478260869563, 'rouge2': 0.1592920353982301, 'rougeL': 0.3130434782608696}

ROUGE RNN+Attention : {'rouge1': 0.08955223880597014, 'rouge2': 0.03076923076923077, 'rougeL': 0.08955223880597014}


#Entraînement du décodeur extractif avec supervision TF-IDF

Dans cette étape, nous entraînons le **décodeur extractif** pour qu’il apprenne à prédire les phrases importantes à partir des représentations RNN + attention.

## Étapes clés :

1. **Création des labels TF-IDF**  
   - Pour chaque article, on sélectionne les `k` phrases les plus importantes selon TF-IDF.  
   - On crée un vecteur binaire de taille `max_sentences` :  
     - `1` pour les phrases sélectionnées  
     - `0` pour les autres (y compris `<PAD>`).

2. **Définition de la perte et de l’optimiseur**  
   - On utilise `BCEWithLogitsLoss` pour la **classification binaire par phrase**.  
   - L’optimiseur est `Adam` avec un taux d’apprentissage `lr=1e-3`.

3. **Boucle d’entraînement**  
   Pour chaque batch :  
   - Les phrases sont encodées avec l’encodeur RNN + attention.  
   - Le décodeur prédit un **score par phrase**.  
   - La perte est calculée par rapport aux labels TF-IDF.  
   - On effectue la rétropropagation et la mise à jour des poids.

4. **Époques et suivi**  
   - La boucle s’exécute sur un nombre d’époques (`epochs=5`).  
   - La perte moyenne par époque est affichée pour suivre l’entraînement.




### **Optimiseurs :**

Les optimiseurs mettent à jour les paramètres de poids pour minimiser la fonction de perte. La fonction de perte agit comme des guides vers le terrain indiquant à l'optimiseur s'il se déplace dans la bonne direction pour atteindre le fond de la vallée, le minimum global.

**Adam** (Adaptive Moment Estimation) est un algorithme d'optimisation, conçu pour trouver efficacement les valeurs optimales des paramètres d'un modèle (ses poids et ses biais) en les mettant à jour de manière itérative sur la base des données d'apprentissage.

Le principe de l'algorithme est qu'il sert à  adapter le taux d'apprentissage à chaque paramètre individuel. Au lieu d'utiliser un taux d'apprentissage unique et fixe pour tous les poids du réseau, Adam calcule un taux d'apprentissage individuel qui s'ajuste au fur et à mesure de l'apprentissage.

 Il tient compte de deux composantes principales : le premier moment (la moyenne des gradients) et le second moment (la variance non centrée des gradients). Cette combinaison lui permet de prendre des pas plus importants pour les paramètres dont les gradients sont cohérents et des pas plus petits pour ceux dont les gradients sont bruyants.

In [11]:
import torch.optim as optim
import torch.nn.functional as F

# Hyperparamètres
epochs = 5
lr = 1e-3

# Optimiseur
optimizer = optim.Adam(decoder.parameters(), lr=lr)
criterion = nn.BCEWithLogitsLoss()

# --- Fonction pour créer les labels TF-IDF ---
def tfidf_labels(sents, k=3, max_sentences=50):
    selected, idxs = tfidf_select(sents, k=k)
    labels = torch.zeros(max_sentences, dtype=torch.float)
    for i in idxs:
        if i < max_sentences:
            labels[i] = 1.0
    return labels

# --- Boucle d'entraînement simplifiée ---
for epoch in range(epochs):
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        batch = batch.to(DEVICE)  # (batch_size, max_sentences, max_len)
        optimizer.zero_grad()
        encoder_outputs = encoder(batch)  # (batch, max_sentences, 2*hidden)
        scores = decoder(encoder_outputs)  # (batch, max_sentences)

        # Créer les labels pour le batch
        batch_labels = []
        for i in range(batch.size(0)):
            # Convertir batch[i] en texte pour TF-IDF
            article_sents = [" ".join([idx2word[idx.item()] for idx in batch[i,j] if idx.item() != word2idx['<PAD>']])
                             for j in range(batch.size(1))]
            article_labels = tfidf_labels(article_sents, k=3, max_sentences=batch.size(1))
            batch_labels.append(article_labels)
        batch_labels = torch.stack(batch_labels).to(DEVICE)

        loss = criterion(scores, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")


Epoch 1/5, Loss: 0.6666
Epoch 2/5, Loss: 0.6372
Epoch 3/5, Loss: 0.6102
Epoch 4/5, Loss: 0.5847
Epoch 5/5, Loss: 0.5608


Après cet entraînement, le décodeur est capable de **sélectionner les phrases importantes directement**, sans avoir besoin de TF-IDF pour la supervision.

#Test et évaluation du décodeur extractif

Dans cette étape, nous générons des résumés extractifs pour plusieurs articles et comparons les performances du modèle **RNN + Attention** avec le **baseline TF-IDF**.

## Étapes clés :

1. **Prétraitement des articles**  
   - Chaque article est découpé en phrases et nettoyé (`preprocess_article`).  
   - Le nombre maximal de phrases est fixé (`max_sentences=50`).

2. **Résumé TF-IDF**  
   - Sélection des `k` phrases les plus représentatives selon le score TF-IDF.

3. **Résumé RNN + Attention**  
   - Encodage des phrases avec le RNN bidirectionnel + attention.  
   - Le décodeur prédit un score pour chaque phrase.  
   - Les `k` phrases avec le score le plus élevé sont sélectionnées.  
   - Les phrases `<PAD>` sont ignorées.

4. **Référence**  
   - Les highlights de l’article servent de résumé de référence.  
   - Découpage simple en phrases.

5. **Calcul des scores ROUGE**  
   - ROUGE-1, ROUGE-2, ROUGE-L sont calculés pour chaque article.  
   - Les scores sont stockés pour TF-IDF et RNN + Attention.

6. **Moyenne des scores**  
   - On calcule les **scores ROUGE moyens** sur l’ensemble des articles testés.  
   - Cela permet d’avoir une **évaluation quantitative globale** des deux méthodes.

---

Cette étape permet de valider **la performance du décodeur entraîné** et de comparer ses résultats avec le baseline TF-IDF, à la fois de manière quantitative (ROUGE moyen) et qualitative (phrases sélectionnées).


In [12]:
from tqdm.auto import tqdm

k = 3  # nombre de phrases à sélectionner
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Fonction ROUGE
def compute_rouge(pred_sents, ref_sents):
    pred_text = " ".join(pred_sents)
    ref_text = " ".join(ref_sents)
    scores = scorer.score(ref_text, pred_text)
    return {k: v.fmeasure for k, v in scores.items()}

# Liste pour stocker les scores
rouge_tfidf_list = []
rouge_rnn_list = []

num_articles = 10  # nombre d'articles à tester (pour le dev)
for i in tqdm(range(num_articles)):
    article = train_articles_subset[i]
    sents = preprocess_article(article, max_sentences=50)

    # TF-IDF baseline
    selected_tfidf, _ = tfidf_select(sents, k=k)

    # RNN + Attention
    batch = torch.tensor(dataset[i]).unsqueeze(0).to(DEVICE)
    encoder_outputs = encoder(batch)
    scores = decoder(encoder_outputs)

    num_real_sents = len(sents)
    topk_indices = torch.topk(scores, k, dim=1).indices[0].cpu().numpy()
    topk_indices_valid = [idx for idx in topk_indices if idx < num_real_sents]
    selected_rnn = [sents[idx] for idx in topk_indices_valid]

    # Référence
    ref = raw['train'][i]['highlights'].split('. ')
    ref = [r.strip() for r in ref if len(r.strip()) > 0]

    # Calcul ROUGE
    rouge_tfidf_list.append(compute_rouge(selected_tfidf, ref))
    rouge_rnn_list.append(compute_rouge(selected_rnn, ref))

# Moyenne des scores ROUGE
def average_rouge(rouge_list):
    avg = {'rouge1':0, 'rouge2':0, 'rougeL':0}
    for r in rouge_list:
        for k in avg.keys():
            avg[k] += r[k]
    for k in avg.keys():
        avg[k] /= len(rouge_list)
    return avg

print("ROUGE moyen TF-IDF :", average_rouge(rouge_tfidf_list))
print("ROUGE moyen RNN+Attention :", average_rouge(rouge_rnn_list))


  0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipython-input-2762355616.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch = torch.tensor(dataset[i]).unsqueeze(0).to(DEVICE)


ROUGE moyen TF-IDF : {'rouge1': 0.1868103885717686, 'rouge2': 0.06592608723812637, 'rougeL': 0.12542857681331315}
ROUGE moyen RNN+Attention : {'rouge1': 0.20430508185523388, 'rouge2': 0.06968192004777932, 'rougeL': 0.14470385200179087}


#Analyse des résultats ROUGE

Après avoir testé le décodeur extractif entraîné sur plusieurs articles, nous obtenons les scores moyens suivants :

| Méthode           | ROUGE-1 | ROUGE-2 | ROUGE-L |
|------------------|---------|---------|---------|
| TF-IDF baseline   | 0.187   | 0.066   | 0.125   |
| RNN + Attention  | 0.136   | 0.041   | 0.099   |

## Observations

1. **Performance actuelle**
   - Le TF-IDF baseline obtient de meilleurs scores pour l’instant.
   - Le décodeur RNN + Attention est encore en phase d’entraînement et n’a pas encore surpassé le TF-IDF.

2. **Explications possibles**
   - Taille du dataset d’entraînement limitée (`train_articles_subset`).
   - Nombre d’époques d’entraînement faible.
   - Poids du décodeur encore insuffisamment ajustés.

3. **Axes d’amélioration**
   - Augmenter le nombre d’articles pour l’entraînement.
   - Entraîner sur plus d’époques et ajuster les hyperparamètres (learning rate, dropout…).
   - Utiliser des résumés TF-IDF plus longs (`k` plus grand) comme supervision.

4. **Analyse qualitative**
   - Comparer visuellement les phrases sélectionnées par TF-IDF et par RNN + Attention.
   - Même avec des scores ROUGE inférieurs, le décodeur peut déjà capturer certaines phrases clés importantes.



In [13]:
from IPython.display import display, Markdown
import ipywidgets as widgets

def test_resume_interface():
    # Zone de texte pour l'article
    text_input = widgets.Textarea(
        value='',
        placeholder='Écris ton article ici...',
        description='Article:',
        layout=widgets.Layout(width='90%', height='200px')
    )

    # Bouton pour générer le résumé
    button = widgets.Button(description="Générer résumé")

    # Affichage des résultats
    output = widgets.Output()

    def on_button_clicked(b):
        with output:
            output.clear_output()
            article_text = text_input.value.strip()
            if article_text == "":
                print("Écris un texte d'abord !")
                return

            # Prétraitement
            sents = preprocess_article(article_text, max_sentences=50)

            # TF-IDF
            selected_tfidf, _ = tfidf_select(sents, k=3)

            # RNN + Attention
            seqs = []
            for sent in sents:
                idxs = [word2idx.get(w.lower(), word2idx['<UNK>']) for w in sent.split()]
                if len(idxs) < 20:
                    idxs += [word2idx['<PAD>']] * (20 - len(idxs))
                else:
                    idxs = idxs[:20]
                seqs.append(idxs)
            batch = torch.tensor([seqs], dtype=torch.long).to(DEVICE)

            encoder_outputs = encoder(batch)
            scores = decoder(encoder_outputs)
            num_real_sents = len(sents)
            topk_indices = torch.topk(scores, 3, dim=1).indices[0].cpu().numpy()
            topk_indices_valid = [sents[i] for i in topk_indices if i < num_real_sents]

            display(Markdown("### Résumé TF-IDF"))
            display(Markdown(" ".join(selected_tfidf)))

            display(Markdown("### Résumé RNN + Attention"))
            display(Markdown(" ".join(topk_indices_valid)))

    button.on_click(on_button_clicked)

    display(text_input, button, output)


In [14]:
test_resume_interface()


Textarea(value='', description='Article:', layout=Layout(height='200px', width='90%'), placeholder='Écris ton …

Button(description='Générer résumé', style=ButtonStyle())

Output()

## Analyse de l'inférence du modèle




---



**Test 1:**

On a testé avec le text de la consigne relative à ce projet.

Voici l'output :     

Résumé TF-IDF



> Sujet 2 Résumé automatique extractif avec attention · Objectif Construire un mini système de résumé automatique de texte résumer un article en quelques phrases · Livrable attendu petit rapport code exemples de résumés avant/après attention qualitatif quantitatif avec ROUGE score · Datasets CNN/DailyMail anglais Wikihow ou articles courts en français presse locale



Résumé RNN + Attention

> Utiliser TF-IDF pour identifier les phrases les plus représentatives · Travail demandé 1 3

   **Analyse output 1:**

 * Le résumé TF-IDF extrait les phrases contenant les mots-clés les plus fréquents et informatifs du texte ("Objectif", "Livrable attendu", "Datasets"). Il restitue les éléments principaux de la consigne, mais sous forme de liste juxtaposée, sans structure ni reformulation. Le résultat est fidèle au contenu, mais manque de clarté et d'organisation : il ne distingue pas les différentes parties (objectifs, livrables, datasets) et ne synthétise pas l'information.

 * Le résumé RNN + Attention est très court et se limite à une phrase partielle, reprenant une consigne technique ("Utiliser TF-IDF pour identifier les phrases les plus représentatives"). Il perd le contexte et n'explique pas l'objectif global du projet. Le résumé est donc incomplet et peu informatif.



---



**Test 2:**

On a testé avec un extrait d'un article de press.

Voici l'output:

Résumé TF-IDF

> https //www.lemonde.fr/en/international/article/2025/09/27/gaza-diplomats-press-trump-to-push-israel-toward-ending-war_6745820_4.html Donald Trump approached the small crowd smiling and self-assured It 's gon na be peace '' he said somewhat cryptically on Friday September 26 as he left the White House to attend the Ryder Cup golf tournament Three days before hosting Israeli Prime Minister Benjamin Netanyahu the president of the United States acted as if the belligerent remarks made by his staunch ally at the United Nations UN podium – calling to `` finish the job '' in Gaza – could be disregarded


Résumé RNN + Attention

> It 's gon na be peace '' he said somewhat cryptically on Friday September 26 as he left the White House to attend the Ryder Cup golf tournament For more information see our Terms and Conditions `` It 's looking like we have a deal on Gaza ...


   **Analyse output 2:**

* Le résumé TF-IDF extrait une longue phrase descriptive, incluant le contexte (Trump, Netanyahu, ONU, Gaza) et une citation. Il restitue fidèlement les faits principaux, mais le résumé est trop dense et inclut des détails secondaires (date, lieu, événement sportif) qui ne sont pas essentiels à la compréhension globale.


* Le résumé RNN + Attention sélectionne une citation marquante et une phrase sur le contexte, mais inclut aussi du bruit ("For more information see our Terms and Conditions"). Le résumé est plus court, mais il perd en cohérence et peut inclure des éléments hors sujet.



---




**Test 3:**

On a testé avec la première section de cet article: [ADAM: A METHOD FOR STOCHASTIC OPTIMIZATION](https://arxiv.org/pdf/1412.6980)


Résumé TF-IDF

> Our method is designed to combine the advantages of two recently popular methods AdaGrad Duchi et al. 2011 which works well with sparse gradients and RMSProp Tieleman Hinton 2012 which works well in on-line and non-stationary settings important connections to these and other stochastic optimization methods are clarified in section 5 Some of Adam ’ s advantages are that the magnitudes of parameter updates are invariant to rescaling of the gradient its stepsizes are approximately bounded by the stepsize hyperparameter it does not require a stationary objective it works with sparse gradients and it naturally performs a form of step size annealing With βt 1 and βt 2 we denote β1 and β2 to the power t. Require α Stepsize Require β1 β2 ∈ 0,1 Exponential decay rates for the moment estimates Require f θ Stochastic objective function with parameters θ Require θ0 Initial parameter vector m0 ←0 Initialize 1st moment vector v0 ←0 Initialize 2nd moment vector t ←0 Initialize timestep while θt not converged do t ←t+1 gt ←∇θft θt−1 Get gradients w.r.t


Résumé RNN + Attention

> For example many objective functions are composed of a sum of subfunctions evaluated at different subsamples of data in this case optimization can be made more efficient by taking gradient steps w.r.t SGD proved itself as an efficient and effective optimization method that was central in many machine learning success stories such as recent advances in deep learning Deng et al. 2013 Krizhevsky et al. 2012 Hinton Salakhutdinov 2006 Hinton et al. 2012a Graves et al. 2013 Our method is designed to combine the advantages of two recently popular methods AdaGrad Duchi et al. 2011 which works well with sparse gradients and RMSProp Tieleman Hinton 2012 which works well in on-line and non-stationary settings important connections to these and other stochastic optimization methods are clarified in section 5


   **Analyse output 3:**

* Le résumé TF-IDF extrait des phrases techniques sur les avantages d'Adam, les méthodes comparées (AdaGrad, RMSProp), et inclut des fragments de pseudo-code et de paramètres. Le résultat est dense, technique, et difficile à lire pour un non-expert. Il mélange explications et éléments algorithmiques sans synthèse.


* Le résumé RNN + Attention sélectionne des phrases sur l'efficacité de SGD, les succès en machine learning, et les avantages d'Adam. Il mélange contexte historique et technique, mais reste un assemblage de phrases extraites, parfois incomplètes ou hors contexte.

---

  **Analyse globale:**

 * Les deux méthodes sont extractives : elles ne reformulent pas, mais sélectionnent des phrases selon leur score ou leur importance perçue.

  * TF-IDF privilégie les phrases riches en mots-clés, ce qui peut inclure des détails secondaires ou du bruit.

  * RNN + Attention améliore la sélection, mais reste limité par la structure du texte et peut inclure des fragments hors contexte ou incomplets.

  * Les deux méthodes sont peu adaptées aux textes techniques ou structurés (code, algorithmes, consignes), car elles ne comprennent pas la logique ni la hiérarchie de l'information.




---




  **Constations:**

  * Les scores ROUGE indiquent que la baseline TF-IDF reste supérieure au modèle RNN+Attention sur les tests actuels.

  * Le système extractif manque d’adaptabilité : il ne peut ignorer des phrases non pertinentes lorsque le texte est très structuré ou contient du bruit (ex : citations, formules, code…).

  * Pour des textes contenant des algorithmes, des formules ou des illustrations complexes, seul un modèle sémantique capable de paraphraser ou de reformuler peut générer des résumés réellement informatifs et cohérents.

  **Origine des limites:**

  * Approche extractive : Cette méthode ne reformule pas, elle sélectionne mécaniquement des phrases « fortes » en termes de mots clés ou de score RNN, sans saisir la structure argumentative ou le raisonnement, ni adapter le contenu aux besoins d’un lecteur.

  * Vocabulaire et structure complexe : Lorsqu’un texte possède des blocs très différents (code, équations, tableaux, citations…), l’encodeur RNN, même avec attention, est limité par la taille du vocabulaire d’entraînement et la difficulté à modéliser des relations complexes ou de long terme.

  * Taille et diversité du dataset d’apprentissage : Avec un sous-ensemble limité d’articles et peu d’époques d’entraînement, le modèle n’apprend pas à généraliser ou à privilégier la cohérence narrative.

  **Améliorations possibles:**

  * Augmenter la taille et la diversité du dataset.

  * Mélanger des formats de texte (expliquer en langage naturel le code ou la formule, relier les explications aux éléments techniques).

  * Envisager des modèles hybrides extractif et génératifs pour combiner sélection et réécriture.

